<a href="https://colab.research.google.com/github/yana-n-k/penjadwalan-kuliah-ag-kelompok-6/blob/main/Penjadwalan_Kuliah_UTS_AG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/yana-n-k/penjadwalan-kuliah-ag-kelompok-6.git
%cd penjadwalan-kuliah-ag-kelompok-6
!ls

Cloning into 'penjadwalan-kuliah-ag-kelompok-6'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
/content/penjadwalan-kuliah-ag-kelompok-6
'data_penjadwalan.xlsx - dosen.csv'
'data_penjadwalan.xlsx - matakuliah.csv'
'data_penjadwalan.xlsx - ruang.csv'
'data_penjadwalan.xlsx - slot_waktu.csv'
 konflik_dpna.csv
 README.md


In [2]:
import argparse
import os
import random
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


def cari_csv(folder, kata):
    cocok = [f for f in os.listdir(folder)
             if f.lower().endswith(".csv") and kata in f.lower()]
    if not cocok:
        raise FileNotFoundError(
            f"Tidak ada file .csv yang namanya mengandung '{kata}' "
            f"di folder '{os.path.abspath(folder)}'")
    cocok.sort(key=lambda f: os.path.getmtime(os.path.join(folder, f)), reverse=True)
    if len(cocok) > 1:
        print(f"CATATAN: ada {len(cocok)} file '{kata}': {cocok} -> dipakai: {cocok[0]}")
    return os.path.join(folder, cocok[0])

In [3]:
def ke_menit(jam):
    h, m = jam.strip().split(":")
    return int(h) * 60 + int(m)


KELOMPOK_PILIHAN = {
    "Penjaminan Mutu Perangkat Lunak": "V",
    "Requirement Engineering":         "V;VII",
    "Algoritma Genetik":               "V",
    "Administrasi Basis Data":         "VII",
    "Deep Learning":                   "VII",
}

JUMAT_MULAI = "11:00"
JUMAT_SELESAI = "12:50"


def _bagian_kelompok(kode):
    if "-" in kode:
        a, b = kode.rsplit("-", 1)
        return a, b
    return kode, None


def _induk(kode):
    return kode.rsplit("-", 1)[0] if "-" in kode else kode


def kelompok_beririsan(a, b):
    for x in a:
        for y in b:
            if x == y or x == _induk(y) or y == _induk(x):
                return True
    return False


class Masalah:

    def __init__(self, folder=".", blokir_jumat=True, silang_ab="auto",
                 kelompok_pilihan=None, paralel_ab=True):
        file_mk = cari_csv(folder, "matakuliah")
        self.mk = pd.read_csv(file_mk, encoding="utf-8-sig")
        self.mk.columns = [str(c).strip().lstrip("\ufeff").lower() for c in self.mk.columns]
        print(f"File matakuliah: {file_mk} | kolom: {list(self.mk.columns)}")
        if kelompok_pilihan is None:
            kelompok_pilihan = KELOMPOK_PILIHAN
        self._terapkan_kelompok_pilihan(kelompok_pilihan)
        self.dosen_df = pd.read_csv(cari_csv(folder, "dosen"))
        self.ruang = pd.read_csv(cari_csv(folder, "ruang"))
        self.slot = pd.read_csv(cari_csv(folder, "slot"))

        self.n = len(self.mk)
        self._cari_pasangan_ab(paralel_ab)
        self.n_slot = len(self.slot)
        self.n_ruang = len(self.ruang)

        self.dosen = self.mk["dosen_id"].tolist()
        self.kelompok = self.mk["kelompok_mhs"].tolist()
        self.kelompok_set = [{x.strip() for x in str(k).split(";") if x.strip() not in ("", "-")}
                             for k in self.kelompok]
        self.mhs = self.mk["jumlah_mhs"].tolist()
        self.kapasitas = self.ruang["kapasitas"].tolist()

        self.kel_bentrok = [[kelompok_beririsan(self.kelompok_set[i], self.kelompok_set[j])
                             for j in range(self.n)] for i in range(self.n)]

        self.sumber_konflik = "DPNA"
        try:
            kf = pd.read_csv(cari_csv(folder, "konflik"))
            self.dpna_semua = [(str(r.id_a), str(r.id_b), int(r.overlap))
                               for r in kf.itertuples()]
        except FileNotFoundError:
            self.sumber_konflik = "manual"
            print("PERINGATAN: konflik_dpna.csv tidak ditemukan di folder data -> memakai "
                  "daftar konflik manual (3 pasangan) dan aturan silang A/B.")
            self.dpna_semua = [("MK16", "MK15", 1), ("MK06", "MK07", 1), ("MK18", "MK08", 2)]
        self.pasangan_konflik_diketahui = self.dpna_semua
        self.ambang_konflik_minor = 3
        id_ke_indeks = {str(v): i for i, v in enumerate(self.mk["id"].astype(str))}
        self.pasangan_konflik_wajib = []
        self.pasangan_konflik_minor = []
        for id_a, id_b, jumlah_overlap in self.dpna_semua:
            if id_a not in id_ke_indeks or id_b not in id_ke_indeks:
                print(f"PERINGATAN: pasangan konflik {id_a}-{id_b} tidak ditemukan "
                      "di data mata kuliah; periksa ID pada matakuliah.csv.")
                continue
            i, j = id_ke_indeks[id_a], id_ke_indeks[id_b]
            if self.kel_bentrok[i][j]:
                continue
            pasangan = (i, j, jumlah_overlap)
            if jumlah_overlap > self.ambang_konflik_minor:
                self.pasangan_konflik_wajib.append(pasangan)
            else:
                self.pasangan_konflik_minor.append(pasangan)

        if silang_ab == "auto":
            silang_ab = "off" if self.sumber_konflik == "DPNA" else "hard"
        self.silang_ab = silang_ab
        sudah_diketahui = {frozenset((a, b)) for a, b, _ in self.pasangan_konflik_diketahui}
        ids = self.mk["id"].astype(str).tolist()
        namas = self.mk["nama"].astype(str).tolist()
        self.pasangan_silang_ab = []
        if silang_ab != "off":
            for i in range(self.n):
                for j in range(i + 1, self.n):
                    if self.kel_bentrok[i][j] or namas[i] == namas[j]:
                        continue
                    if frozenset((ids[i], ids[j])) in sudah_diketahui:
                        continue
                    saudara = any(
                        _bagian_kelompok(a)[1] and _bagian_kelompok(b)[1]
                        and _bagian_kelompok(a)[0] == _bagian_kelompok(b)[0]
                        and a != b
                        for a in self.kelompok_set[i] for b in self.kelompok_set[j])
                    if saudara:
                        self.pasangan_silang_ab.append((i, j))

        mulai = self.slot["jam_mulai"].apply(ke_menit).tolist()
        selesai = self.slot["jam_selesai"].apply(ke_menit).tolist()
        hari = self.slot["hari"].tolist()
        self.bentrok = [
            [
                hari[a] == hari[b]
                and mulai[a] < selesai[b]
                and mulai[b] < selesai[a]
                for b in range(self.n_slot)
            ]
            for a in range(self.n_slot)
        ]

        self.terblokir = set()
        if blokir_jumat:
            mulai_blokir, selesai_blokir = ke_menit(JUMAT_MULAI), ke_menit(JUMAT_SELESAI)
            for idx, row in self.slot.iterrows():
                mulai_slot = ke_menit(str(row["jam_mulai"]))
                selesai_slot = ke_menit(str(row["jam_selesai"]))
                if (str(row["hari"]).strip().lower() == "jumat"
                        and mulai_slot < selesai_blokir
                        and mulai_blokir < selesai_slot):
                    self.terblokir.add(idx)

        self._cek_data()

    def _cari_pasangan_ab(self, aktif):
        self.pasangan_ab = []
        if not aktif:
            return
        kelompok = {}
        for i, row in self.mk.iterrows():
            kls = str(row["kelas"]).strip().upper()
            if kls in ("", "-", "NAN"):
                continue
            sem = str(row["semester"]).strip() if "semester" in self.mk.columns else ""
            kunci = (str(row["nama"]).strip().lower(), sem)
            kelompok.setdefault(kunci, []).append(i)
        for (nama, _), idx in kelompok.items():
            if len(idx) < 2:
                continue
            idx = sorted(idx, key=lambda i: str(self.mk.loc[i, "kelas"]).strip().upper())
            pemimpin = idx[0]
            for f in idx[1:]:
                if self.mk.loc[f, "dosen_id"] == self.mk.loc[pemimpin, "dosen_id"]:
                    print(f"PERINGATAN: {self.mk.loc[pemimpin, 'id']} dan {self.mk.loc[f, 'id']} "
                          f"({nama}) diajar dosen yang sama -> tidak bisa sejadwal, dilewati.")
                    continue
                self.pasangan_ab.append((f, pemimpin))
        print(f"Pasangan kelas A/B yang disejadwalkan bersamaan: {len(self.pasangan_ab)}")

    def _terapkan_kelompok_pilihan(self, kelompok_pilihan):
        self.mk["kelompok_mhs"] = self.mk["kelompok_mhs"].astype(str)
        nama = self.mk["nama"].astype(str).str.strip().str.lower()
        for nama_mk, kel in kelompok_pilihan.items():
            mask = nama == nama_mk.strip().lower()
            if not mask.any():
                print(f"PERINGATAN: matkul pilihan '{nama_mk}' tidak ditemukan di CSV.")
                continue
            lama = self.mk.loc[mask, "kelompok_mhs"].iloc[0]
            self.mk.loc[mask, "kelompok_mhs"] = kel
            print(f"Kelompok '{nama_mk}': '{lama}' -> '{kel}'")

    def _cek_data(self):
        for i in range(self.n):
            if not self.kelompok_set[i]:
                print(f"PERINGATAN: kelas {self.mk.loc[i, 'id']} ({self.mk.loc[i, 'nama']}) "
                      "belum punya kelompok mahasiswa yang valid -> hanya dicek bentrok "
                      "dosen/ruang, bukan bentrok mahasiswa. Isi di KELOMPOK_PILIHAN atau CSV.")
        max_kap = max(self.kapasitas)
        for i in range(self.n):
            if self.mhs[i] > max_kap:
                print(f"PERINGATAN: kelas {self.mk.loc[i, 'id']} "
                      f"({self.mhs[i]} mhs) melebihi kapasitas ruang terbesar.")
        id_dosen = set(self.dosen_df["id"])
        for d in set(self.dosen):
            if d not in id_dosen:
                print(f"PERINGATAN: dosen_id {d} tidak ada di file dosen.")

In [4]:
def hitung_pelanggaran(m, krom):
    r = {
        "dosen": 0, "ruang": 0, "kelompok": 0,
        "peserta_dpna_wajib": 0, "peserta_minor": 0,
        "silang_ab_wajib": 0, "silang_ab_soft": 0,
        "kapasitas": 0, "slot_terblokir": 0,
        "ab_beda_slot": 0
    }
    for i in range(m.n):
        s_i, r_i = krom[i]
        if m.mhs[i] > m.kapasitas[r_i]:
            r["kapasitas"] += 1
        if s_i in m.terblokir:
            r["slot_terblokir"] += 1
        for j in range(i + 1, m.n):
            s_j, r_j = krom[j]
            if m.bentrok[s_i][s_j]:
                if m.dosen[i] == m.dosen[j]:
                    r["dosen"] += 1
                if r_i == r_j:
                    r["ruang"] += 1
                if m.kel_bentrok[i][j]:
                    r["kelompok"] += 1

    for f, l in m.pasangan_ab:
        if krom[f][0] != krom[l][0]:
            r["ab_beda_slot"] += 1

    for i, j, jumlah_overlap in m.pasangan_konflik_wajib:
        if m.bentrok[krom[i][0]][krom[j][0]]:
            r["peserta_dpna_wajib"] += 1

    for i, j, jumlah_overlap in m.pasangan_konflik_minor:
        if m.bentrok[krom[i][0]][krom[j][0]]:
            r["peserta_minor"] += 1

    kunci = "silang_ab_wajib" if m.silang_ab == "hard" else "silang_ab_soft"
    for i, j in m.pasangan_silang_ab:
        if m.bentrok[krom[i][0]][krom[j][0]]:
            r[kunci] += 1

    return r


def fitness(m, krom):
    pel = hitung_pelanggaran(m, krom)
    soft = ("peserta_minor", "silang_ab_soft")
    wajib = sum(v for k, v in pel.items() if k not in soft)
    minor = sum(pel[k] for k in soft)
    skor = wajib + 0.05 * minor
    return 1.0 / (1.0 + skor), wajib

In [5]:
def sinkron_ab(m, krom):
    for f, l in m.pasangan_ab:
        krom[f] = (krom[l][0], krom[f][1])
    return krom


def buat_individu(m, rng):
    """Kromosom acak: gen ke-i = (slot, ruang) untuk kelas ke-i."""
    krom = [(rng.randrange(m.n_slot), rng.randrange(m.n_ruang))
            for _ in range(m.n)]
    return sinkron_ab(m, krom)


def seleksi_roulette(pop, fit, rng):
    """Roulette wheel: peluang terpilih sebanding dengan fitness."""
    return rng.choices(pop, weights=fit, k=1)[0]


def seleksi_turnamen(pop, fit, rng, k=3):
    """Turnamen: ambil k individu acak, fitness tertinggi menang."""
    kandidat = rng.sample(range(len(pop)), k)
    menang = max(kandidat, key=lambda i: fit[i])
    return pop[menang]


def crossover_satu_titik(a, b, rng):
    """Potong di satu titik, tukar bagian belakang."""
    titik = rng.randint(1, len(a) - 1)
    return a[:titik] + b[titik:], b[:titik] + a[titik:]


def crossover_uniform(a, b, rng):
    """Tiap gen diambil dari ayah atau ibu dengan peluang 50:50."""
    c1, c2 = [], []
    for ga, gb in zip(a, b):
        if rng.random() < 0.5:
            c1.append(ga)
            c2.append(gb)
        else:
            c1.append(gb)
            c2.append(ga)
    return c1, c2


def mutasi(m, krom, pm, rng):
    """Tiap gen memiliki peluang pm dimutasi: slot, ruang, atau keduanya diganti acak."""
    hasil = []
    for (s, r) in krom:
        if rng.random() < pm:
            pilihan = rng.random()
            if pilihan < 0.4:
                s = rng.randrange(m.n_slot)
            elif pilihan < 0.8:
                r = rng.randrange(m.n_ruang)
            else:
                s = rng.randrange(m.n_slot)
                r = rng.randrange(m.n_ruang)
        hasil.append((s, r))
    return sinkron_ab(m, hasil)

In [6]:
def jalankan_ga(m, seed, cfg):
    rng = random.Random(seed)
    mulai_waktu = time.time()

    pilih = seleksi_roulette if cfg.seleksi == "roulette" else seleksi_turnamen
    kawin = crossover_uniform if cfg.crossover == "uniform" else crossover_satu_titik

    pop = [buat_individu(m, rng) for _ in range(cfg.pop)]
    riwayat_best, riwayat_avg = [], []
    pertama_valid = None

    for gen in range(cfg.gen + 1):
        nilai = [fitness(m, k) for k in pop]
        fit = [v[0] for v in nilai]
        viol = [v[1] for v in nilai]
        urut = sorted(range(len(pop)), key=lambda i: fit[i], reverse=True)

        best_i = urut[0]
        riwayat_best.append(fit[best_i])
        riwayat_avg.append(sum(fit) / len(fit))
        if viol[best_i] == 0 and pertama_valid is None:
            pertama_valid = gen

        if gen == cfg.gen or (cfg.berhenti_jika_valid and viol[best_i] == 0):
            break


        baru = [list(pop[i]) for i in urut[:cfg.elit]]
        while len(baru) < cfg.pop:
            p1 = pilih(pop, fit, rng)
            p2 = pilih(pop, fit, rng)
            if rng.random() < cfg.pc:
                c1, c2 = kawin(p1, p2, rng)
            else:
                c1, c2 = list(p1), list(p2)
            baru.append(mutasi(m, c1, cfg.pm, rng))
            if len(baru) < cfg.pop:
                baru.append(mutasi(m, c2, cfg.pm, rng))
        pop = baru

    return {
        "seed": seed,
        "best_fit": fit[best_i],
        "best_viol": viol[best_i],
        "best_minor": (lambda p: p["peserta_minor"] + p["silang_ab_soft"])(
            hitung_pelanggaran(m, pop[best_i])),
        "avg_fit_akhir": sum(fit) / len(fit),
        "generasi_valid_pertama": pertama_valid,
        "generasi_berjalan": gen,
        "waktu_detik": round(time.time() - mulai_waktu, 2),
        "riwayat_best": riwayat_best,
        "riwayat_avg": riwayat_avg,
        "krom": pop[best_i],
    }

In [7]:
def buat_tabel(m, krom):
    nama_dosen = dict(zip(m.dosen_df["id"], m.dosen_df["nama"]))
    hari_urut = m.slot["hari"].drop_duplicates().tolist()
    slot_urut = sorted(m.slot["slot_ke"].unique())

    sel = {}
    detail = []
    for i, (s, r) in enumerate(krom):
        h = m.slot.loc[s, "hari"]
        k = m.slot.loc[s, "slot_ke"]
        jam = f'{m.slot.loc[s, "jam_mulai"]}-{m.slot.loc[s, "jam_selesai"]}'
        kls = m.mk.loc[i]
        teks = (f'{kls["id"]} {kls["nama"]} ({kls["kelas"]})\n'
                f'{m.ruang.loc[r, "nama"]}')
        sel.setdefault((h, k), []).append(teks)
        detail.append({
            "id_mk": kls["id"], "mata_kuliah": kls["nama"], "kelas": kls["kelas"],
            "dosen": nama_dosen.get(kls["dosen_id"], kls["dosen_id"]),
            "kelompok_mhs": kls["kelompok_mhs"], "jumlah_mhs": kls["jumlah_mhs"],
            "hari": h, "slot_ke": k, "jam": jam,
            "ruang": m.ruang.loc[r, "nama"], "kapasitas_ruang": m.kapasitas[r],
        })

    baris = {}
    for k in slot_urut:
        contoh = m.slot[m.slot["slot_ke"] == k].iloc[0]
        label = f'Slot {k} ({contoh["jam_mulai"]}-{contoh["jam_selesai"]})'
        baris[label] = {h: "\n\n".join(sel.get((h, k), [])) for h in hari_urut}
    grid = pd.DataFrame(baris).T
    grid.index.name = "Slot"

    detail = pd.DataFrame(detail)
    detail["_h"] = detail["hari"].map({h: n for n, h in enumerate(hari_urut)})
    detail = (detail.sort_values(["_h", "slot_ke", "ruang"])
              .drop(columns="_h").reset_index(drop=True))
    return grid, detail


def simpan_hasil(m, hasil_runs, folder_out):
    os.makedirs(folder_out, exist_ok=True)

    valid_runs = [x for x in hasil_runs if x["best_viol"] == 0]
    kandidat = valid_runs if valid_runs else hasil_runs
    terbaik = max(kandidat, key=lambda x: (
        x["best_fit"],
        -(x["generasi_valid_pertama"] if x["generasi_valid_pertama"] is not None else 10**9)))
    grid, detail = buat_tabel(m, terbaik["krom"])

    grid.to_csv(os.path.join(folder_out, "jadwal_terbaik_grid.csv"), encoding="utf-8-sig")
    detail.to_csv(os.path.join(folder_out, "jadwal_terbaik_detail.csv"),
                  index=False, encoding="utf-8-sig")

    pelanggaran_terbaik = hitung_pelanggaran(m, terbaik["krom"])
    pd.DataFrame([{"jenis_kendala": k, "jumlah": v,
                   "status": ("TOLERANSI MINOR" if k in ("peserta_minor", "silang_ab_soft") and v > 0
                              else ("AMAN" if v == 0 else "PELANGGARAN"))}
                  for k, v in pelanggaran_terbaik.items()]).to_csv(
        os.path.join(folder_out, "validasi_pelanggaran.csv"),
        index=False, encoding="utf-8-sig")

    path_xlsx = os.path.join(folder_out, "jadwal_terbaik_grid.xlsx")
    with pd.ExcelWriter(path_xlsx, engine="openpyxl") as w:
        grid.to_excel(w, sheet_name="Jadwal (hari x slot)")
        detail.to_excel(w, sheet_name="Detail", index=False)
        from openpyxl.styles import Alignment
        ws = w.sheets["Jadwal (hari x slot)"]
        ws.column_dimensions["A"].width = 24
        for kol in "BCDEF":
            ws.column_dimensions[kol].width = 34
        for baris in ws.iter_rows(min_row=2):
            ws.row_dimensions[baris[0].row].height = 95
            for c in baris:
                c.alignment = Alignment(wrap_text=True, vertical="top")

    ringkas = pd.DataFrame([{k: v for k, v in r.items()
                             if k not in ("riwayat_best", "riwayat_avg", "krom")}
                            for r in hasil_runs])
    ringkas.to_csv(os.path.join(folder_out, "ringkasan_run.csv"),
                   index=False, encoding="utf-8-sig")

    plt.figure(figsize=(9, 5))
    for r in hasil_runs:
        plt.plot(r["riwayat_best"], color="lightgray", linewidth=1)
    plt.plot([], [], color="lightgray", label="Fitness terbaik (run lain)")
    plt.plot(terbaik["riwayat_best"], color="tab:blue", linewidth=2,
             label=f'Fitness terbaik (seed {terbaik["seed"]})')
    plt.plot(terbaik["riwayat_avg"], color="tab:orange", linewidth=1.5,
             label=f'Fitness rata-rata populasi (seed {terbaik["seed"]})')
    plt.xlabel("Generasi")
    plt.ylabel("Fitness")
    plt.title("Grafik Konvergensi Fitness per Generasi")
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(folder_out, "konvergensi.png"), dpi=150)
    plt.close()

    return terbaik, grid, detail, ringkas

In [8]:
def audit_matkul_pilihan(m, krom):
    print("\n=== AUDIT MATKUL PILIHAN ===")
    for nama_mk in KELOMPOK_PILIHAN:
        idx = [i for i in range(m.n) if str(m.mk.loc[i, "nama"]).lower() == nama_mk.lower()]
        for i in idx:
            bentrok = [m.mk.loc[j, "id"] for j in range(m.n)
                       if j != i and m.kel_bentrok[i][j]
                       and m.bentrok[krom[i][0]][krom[j][0]]]
            s, r = krom[i]
            print(f"{m.mk.loc[i, 'id']} {nama_mk} [{m.kelompok[i]}] -> "
                  f"{m.slot.loc[s, 'hari']} {m.slot.loc[s, 'jam_mulai']}-{m.slot.loc[s, 'jam_selesai']}, "
                  f"bentrok dengan: {bentrok if bentrok else 'tidak ada'}")


def audit_pasangan_ab(m, krom):
    for f, l in m.pasangan_ab:
        sl, rl = krom[l]
        sf, rf = krom[f]
        ok = (sl == sf) and (rl != rf)
        print(f"{m.mk.loc[l, 'id']} ({m.mk.loc[l, 'nama']} {m.mk.loc[l, 'kelas']}) "
              f"& {m.mk.loc[f, 'id']} ({m.mk.loc[f, 'kelas']}): "
              f"{m.slot.loc[sl, 'hari']} {m.slot.loc[sl, 'jam_mulai']}-{m.slot.loc[sl, 'jam_selesai']}, "
              f"ruang {m.ruang.loc[rl, 'nama']} / {m.ruang.loc[rf, 'nama']} "
              f"-> {'OK' if ok else 'TIDAK SESUAI'}")


def audit_dpna(m, krom):
    if m.sumber_konflik != "DPNA":
        return
    id_ke_indeks = {str(v): i for i, v in enumerate(m.mk["id"].astype(str))}
    bentrok = []
    for id_a, id_b, ov in m.dpna_semua:
        i, j = id_ke_indeks.get(id_a), id_ke_indeks.get(id_b)
        if i is None or j is None:
            continue
        if m.bentrok[krom[i][0]][krom[j][0]]:
            bentrok.append((id_a, id_b, ov))
    print("\n=== AUDIT BENTROK MAHASISWA (berdasarkan NIM di DPNA) ===")
    if not bentrok:
        print("Tidak ada mahasiswa yang bentrok jadwal.")
        return
    for a, b, ov in sorted(bentrok, key=lambda x: -x[2]):
        print(f"{a} <-> {b}: {ov} mahasiswa")
    print(f"Total: {len(bentrok)} pasangan kelas bentrok, "
          f"{sum(x[2] for x in bentrok)} mahasiswa (dijumlah per pasangan)")


def main():
    p = argparse.ArgumentParser(description="AG penjadwalan kuliah (UTS) - kelompok_mhs dan toleransi konflik minor")
    p.add_argument("--data", default=".", help="folder berisi 4 file CSV")
    p.add_argument("--out", default="hasil", help="folder output")
    p.add_argument("--runs", type=int, default=10, help="jumlah run (seed berbeda)")
    p.add_argument("--seed-awal", type=int, default=1)
    p.add_argument("--pop", type=int, default=100, help="ukuran populasi")
    p.add_argument("--gen", type=int, default=100, help="jumlah generasi")
    p.add_argument("--pc", type=float, default=0.8, help="peluang crossover")
    p.add_argument("--pm", type=float, default=0.05, help="peluang mutasi per gen")
    p.add_argument("--elit", type=int, default=2, help="jumlah individu elitisme")
    p.add_argument("--seleksi", choices=["tournament", "roulette"], default="tournament")
    p.add_argument("--crossover", choices=["uniform", "onepoint"], default="uniform")
    p.add_argument("--tanpa-blokir-jumat", dest="blokir_jumat", action="store_false",
                   help="izinkan kuliah saat jendela Jumatan (default: slot yang beririsan "
                        "dengan Jumat 11.00-12.50 dilarang)")
    p.add_argument("--silang-ab", choices=["auto", "off", "soft", "hard"], default="auto",
                   help="matkul wajib kelas A vs B (kelompok saudara) sejadwal: "
                        "auto=off jika ada konflik_dpna.csv, selain itu hard (default); "
                        "off=abaikan, soft=penalti kecil, hard=dilarang")
    p.add_argument("--tanpa-paralel-ab", dest="paralel_ab", action="store_false",
                   help="matikan aturan kelas A dan B harus sejadwal (default: aktif)")
    p.add_argument("--berhenti-jika-valid", dest="berhenti_jika_valid", action="store_true",
                   help="berhenti begitu solusi valid pertama ketemu "
                        "(default: lanjut sampai --gen)")
    cfg, _ = p.parse_known_args()

    m = Masalah(cfg.data, blokir_jumat=cfg.blokir_jumat, silang_ab=cfg.silang_ab,
                paralel_ab=cfg.paralel_ab)
    print(f"Data: {m.n} kelas, {len(m.dosen_df)} dosen, "
          f"{m.n_ruang} ruang, {m.n_slot} slot")
    print(f"Parameter: pop={cfg.pop} gen={cfg.gen} pc={cfg.pc} pm={cfg.pm} "
          f"elit={cfg.elit} seleksi={cfg.seleksi} crossover={cfg.crossover} "
          f"blokir_jumat={cfg.blokir_jumat} silang_ab={m.silang_ab}\n")

    hasil_runs = []
    for k in range(cfg.runs):
        seed = cfg.seed_awal + k
        r = jalankan_ga(m, seed, cfg)
        hasil_runs.append(r)
        print(f"Run {k + 1:>2} (seed {seed:>3}): fitness terbaik={r['best_fit']:.4f} "
              f"pelanggaran={r['best_viol']} valid pertama di generasi="
              f"{r['generasi_valid_pertama']} waktu={r['waktu_detik']}s")

    terbaik, grid, detail, ringkas = simpan_hasil(m, hasil_runs, cfg.out)

    print("\n=== RINGKASAN ===")
    print(f"Fitness terbaik (semua run): {ringkas['best_fit'].max():.4f}")
    print(f"Rata-rata fitness terbaik per run: {ringkas['best_fit'].mean():.4f}")
    print(f"Rata-rata fitness populasi di generasi terakhir: "
          f"{ringkas['avg_fit_akhir'].mean():.4f}")
    valid = ringkas["generasi_valid_pertama"].dropna()
    print(f"Run yang menemukan solusi valid: {len(valid)} dari {cfg.runs}")
    if len(valid):
        print(f"Rata-rata generasi valid pertama: {valid.mean():.1f}")

    pelanggaran_akhir = hitung_pelanggaran(m, terbaik["krom"])
    print(f"\nRincian pelanggaran jadwal terbaik (seed {terbaik['seed']}): "
          f"{pelanggaran_akhir}")
    if terbaik["best_viol"] == 0:
        print("STATUS: VALID — semua hard constraints yang dimodelkan terpenuhi.")
    else:
        print("STATUS: BELUM VALID — masih ada hard constraints yang dilanggar. "
              "Jangan gunakan sebagai jadwal final; tambah generasi/populasi atau "
              "periksa apakah data dan batasan memungkinkan solusi.")
    audit_matkul_pilihan(m, terbaik["krom"])
    audit_pasangan_ab(m, terbaik["krom"])
    audit_dpna(m, terbaik["krom"])
    print(f"\nFile hasil ada di folder '{cfg.out}/'")
    print("\nJadwal terbaik (detail):")
    print(detail.to_string(index=False))

In [9]:
if __name__ == "__main__":
    main()

File matakuliah: ./data_penjadwalan.xlsx - matakuliah.csv | kolom: ['id', 'nama', 'kelas', 'semester', 'sks', 'dosen_id', 'jumlah_mhs', 'kelompok_mhs']
Kelompok 'Penjaminan Mutu Perangkat Lunak': '-' -> 'V'
Kelompok 'Requirement Engineering': '-' -> 'V;VII'
Kelompok 'Algoritma Genetik': '-' -> 'V'
Kelompok 'Administrasi Basis Data': '-' -> 'VII'
Kelompok 'Deep Learning': '-' -> 'VII'
Pasangan kelas A/B yang disejadwalkan bersamaan: 7
Data: 20 kelas, 8 dosen, 5 ruang, 25 slot
Parameter: pop=100 gen=100 pc=0.8 pm=0.05 elit=2 seleksi=tournament crossover=uniform blokir_jumat=True silang_ab=off

Run  1 (seed   1): fitness terbaik=1.0000 pelanggaran=0 valid pertama di generasi=7 waktu=1.0s
Run  2 (seed   2): fitness terbaik=1.0000 pelanggaran=0 valid pertama di generasi=7 waktu=1.11s
Run  3 (seed   3): fitness terbaik=1.0000 pelanggaran=0 valid pertama di generasi=5 waktu=1.69s
Run  4 (seed   4): fitness terbaik=1.0000 pelanggaran=0 valid pertama di generasi=11 waktu=1.57s
Run  5 (seed   5)